In [ ]:
import sys
sys.path.append('../src')

In [ ]:
import io
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np 

# Montly data loader

In [ ]:
def load_month(year,month, data_dir = "../data"):
    path = f"{data_dir}/A-{year}/F{year}-{month:02d}.xls" # 02d fills with zeros unitl 2 digits
    df_raw = pd.read_excel(path)

    # separating total summary rows
    # past logic:  
    # mask_total = (df_raw["Año"]=="Total") 
    # df = df_raw[~mask_total]
    mask = (df_raw["Año"]==year) & (df_raw["Mes"] == month)
    df = df_raw[mask]

    # droping usless columns
    cols_to_drop = ['Hora', 'T. máx.', 'Hora.1', 'T. mín.', 'Hora.2', 'HR. mx.', 'Hora.3', 'HR. mn.', 'Hora.4']
    df = df.drop(columns= cols_to_drop)

    #renaming
    rename_map = {
        "Año": "year",
        "Mes": "month",
        "Día": "day",
        "H": "H",
        "P (mm)": "rain_mm",
        "T (°C)": "temp",
        "HR %": "humidity",
        "Tens. V": "vapor_pressure",
        "Rocío": "dew_point",
    }
    df = df.rename(columns=rename_map)

    # parsing H values
    def parse_hour(raw_value):
        chain = str(int(raw_value)).zfill(4)
        hours = chain[:2]
        minutes = chain[2:]
        return f"{hours}:{minutes}"

    df["H"] = df["H"].apply(parse_hour)

    # changing 24:00 to 00:00 (pending: adding a day)
    mask_2400 = df["H"] == "24:00"
    df.loc[mask_2400, "H"] = "00:00"

    # creating a timestamp
    df["hour"] = df["H"].str[:2]
    df["minute"] = df["H"].str[3:5]
    df["second"] = 0
    df["date"]=pd.to_datetime(df[["year","month","day","hour","minute","second"]])

    # changing the order 
    new_order = [
    "date",
    "temp",
    "humidity",
    "vapor_pressure",
    "dew_point",
    "rain_mm",
    "H",
    "year",
    "month",
    "day",
    "hour",
    "minute",
    "second"
    ]

    df = df[new_order]

    # adding a day
    df.loc[mask_2400,"date"] += pd.Timedelta(days=1)

    return df


In [ ]:
df = load_month(2001,2)

In [ ]:
df

In [ ]:
import sys
print(sys.path)

In [ ]:
from loaders import load_month


df = load_month(2001,3)
df.head()

# Range data loader

In [ ]:
def load_range(start_year, start_month, end_year, end_month, data_dir = "../data"):
    monthly_dfs = []
    start = f"{start_year}-{start_month:02d}-01"
    end = f"{end_year}-{end_month:02d}-01"
    months = pd.date_range(start,end,freq = "MS")
    for i in months:
    #try:
        year = i.year
        month = i.month 
        monthly_dfs.append(load_month(year, month))
    #except Exception as e:
        #print(f"Error loading {month} {year}")
        #raise
    combined = pd.concat(monthly_dfs).reset_index(drop = True)
    return combined

In [ ]:
df = load_range(2001,1,2011,12) #this function treats it like a closed interval

In [ ]:
print(df.shape)
print(df.dtypes)

print(df["day"].max())
print(df["date"].min(), df["date"].max())

Note: Some files contain measurements recorded at `24:00`. These timestamps are normalized to `00:00` of the following day, so the final timestamp can slightly exceed the requested date range.

### Errors to solve: (solved, that's why the previous cell worked)

Error loading 1 2002

Error loading 2 2006

## Corrupt value in `F2002-01.xls`

Data from February is included inside the Jaunary file.

| Index | Año | Mes | Día | H |
|---:|---:|---:|---:|---:|
| 4492 | 2002 | 1 | 31 | 2350.0 |
| 4493 | 2002 | 1 | 31 | 2400.0 |
| 4494 | Total |  | 31 | NaN |
| 4495 | 2002 | 2 | 1 | 10.0 |
| 4496 | 2002 | 2 | 1 | 20.0 |
| ... | ... | ... | ... | ... |
| 4554 | 2002 | 2 | 1 | 1000.0 |
| 4555 | 2002 | 2 | 1 | 1010.0 |
| 4556 | 2002 | 2 | 1 | 1020.0 |
| 4557 | 2002 | 2 | 1 | 1030.0 |
| 4558 | 2002 | 2 | 32 | 1040.0 |


`Día = 32` is incorrect.

The loader should filter records using `year` and `month` parameters before creating the datetime column.

In [ ]:
df_raw = pd.read_excel("../data/A-2002/F2002-01.xls")

df_raw

In [ ]:

df_raw = pd.read_excel("../data/A-2002/F2002-01.xls")

print(df_raw[["Año", "Mes", "Día", "H"]].tail((4557-4490)))

## Corrupt value in `F2006-02.xls`


There is data from March in February records and a corrupt `Día = 60`.


| Index | Año | Mes | Día | H |
|---:|---:|---:|---:|---:|
| 4059 | Total |  | 28 | NaN |
| 4060 | 2006 | 3 | 1 | 10.0 |
| 4061 | 2006 | 3 | 1 | 20.0 |
| 4062 | 2006 | 3 | 1 | 30.0 |
| 4063 | 2006 | 3 | 1 | 40.0 |
| ... | ... | ... | ... | ... |
| 4165 | 2006 | 3 | 1 | 1740.0 |
| 4166 | 2006 | 3 | 1 | 1750.0 |
| 4167 | 2006 | 3 | 1 | 1800.0 |
| 4168 | 2006 | 3 | 1 | 1810.0 |
| 4169 | 2006 | 3 | 60 | 1820.0 |

In [ ]:
df_raw = pd.read_excel("../data/A-2006/F2006-02.xls")

print(df_raw[["Año", "Mes", "Día", "H"]].tail((4557-4490)))

Loader test: (restart kernel and execute only the first cell)

In [ ]:
from loaders import load_range

df = load_range(2002,3,2007,8)

print(df.shape)

In [ ]:
df